In [131]:
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import SVC


%load_ext autoreload
%autoreload 2
import importlib
import sys
sys.path.append("../src/")
import modele
importlib.reload(modele)
import nettoyage
importlib.reload(nettoyage)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'nettoyage' from 'c:\\Users\\anton\\Downloads\\A3\\analyse données\\Winunmax\\notebooks\\../src\\nettoyage.py'>

In [132]:
df_valeur_joueurs = pd.read_csv("../data/brute/player_valuation_before_season.csv")
df_composition_equipes = pd.read_csv("../data/brute/game_lineups.csv")
df_resultat_matchs = pd.read_csv("../data/brute/matchs_2013_2022.csv")

df = pd.read_csv("../data/brute/match_2023.csv")

df_valeur_joueurs["date"] = pd.to_datetime(df_valeur_joueurs["date"])
df_composition_equipes["date"] = pd.to_datetime(df_composition_equipes["date"])
df_resultat_matchs["date"] = pd.to_datetime(df_resultat_matchs["date"])
df["date"] = pd.to_datetime(df["date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   game_id                 270 non-null    int64         
 1   date                    270 non-null    datetime64[ns]
 2   home_club_id            270 non-null    int64         
 3   away_club_id            270 non-null    int64         
 4   home_club_manager_name  270 non-null    object        
 5   away_club_manager_name  270 non-null    object        
 6   stadium                 270 non-null    object        
 7   attendance              266 non-null    float64       
 8   referee                 270 non-null    object        
 9   home_club_name          270 non-null    object        
 10  away_club_name          270 non-null    object        
 11  competition_type        270 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(3), object

C:\Users\anton\AppData\Local\Temp\ipykernel_56468\2729266873.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_composition_equipes = pd.read_csv("../data/brute/game_lineups.csv")


In [133]:
df_resultat_matchs = df_resultat_matchs[["home_club_name","away_club_name","game_id","date","season","home_club_id","away_club_id","home_club_position","away_club_position","results","home_club_goals","away_club_goals"]]
df_resultat_matchs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4078 entries, 0 to 4077
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   home_club_name      4078 non-null   object        
 1   away_club_name      4078 non-null   object        
 2   game_id             4078 non-null   int64         
 3   date                4078 non-null   datetime64[ns]
 4   season              4078 non-null   int64         
 5   home_club_id        4078 non-null   int64         
 6   away_club_id        4078 non-null   int64         
 7   home_club_position  4078 non-null   float64       
 8   away_club_position  4078 non-null   float64       
 9   results             4078 non-null   int64         
 10  home_club_goals     4078 non-null   int64         
 11  away_club_goals     4078 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int64(7), object(2)
memory usage: 382.4+ KB


In [134]:
df = df.drop(columns=["home_club_manager_name","away_club_manager_name","stadium","attendance","referee","competition_type"])
df['season'] = 2023

In [135]:
df_2023 = pd.concat([df_resultat_matchs, df], axis=0, ignore_index=True, sort=False)
df_2023.tail(3)

,home_club_name,away_club_name,game_id,date,season,home_club_id,away_club_id,home_club_position,away_club_position,results,home_club_goals,away_club_goals
4345,Paris Saint-Germain Football Club,Stade Rennais Football Club,4094776,2024-02-25,2023,583,273,NaN,NaN,NaN,NaN,NaN
4346,Toulouse Football Club,Olympique Gymnaste Club Nice Côte d'Azur,4094786,2024-03-03,2023,415,417,NaN,NaN,NaN,NaN,NaN
4347,Racing Club de Strasbourg Alsace,Stade brestois 29,4094777,2024-02-24,2023,667,3911,NaN,NaN,NaN,NaN,NaN


In [136]:
#Ajout ratio valorisation home par away
df_2023 = nettoyage.calcul_difference_valorisation(df_2023,df_composition_equipes,df_valeur_joueurs)
df_2023 = nettoyage.calcul_ratio_valorisation(df_2023,df_composition_equipes,df_valeur_joueurs)
df_2023 = nettoyage.calcul_log_ratio_valorisation(df_2023,df_composition_equipes,df_valeur_joueurs)
#df_2023 = nettoyage.calcul_valorisation_flexible(df_2023,df_composition_equipes,df_valeur_joueurs,methode="diff")
#df_2023 = nettoyage.calcul_valorisation_flexible(df_2023,df_composition_equipes,df_valeur_joueurs,methode="ratio")
#df_2023 = nettoyage.calcul_valorisation_flexible(df_2023,df_composition_equipes,df_valeur_joueurs,methode="log_ratio")

In [137]:
df_2023 = nettoyage.remplir_valeurs_2023_depuis_2022(df_2023)

In [138]:
df_2023["difference classement"] = df_2023["home_club_position"]-df_2023["away_club_position"]

#Ajout winrate
nb_annes = 1
df_2023 = nettoyage.calcul_winrate_historique(df_2023,nb_annes)
df_2023["difference winrate home-away "+str(nb_annes)+" ans"] = df_2023["winrate_home "+str(nb_annes)+" ans"]-df_2023["winrate_away "+str(nb_annes)+" ans"]
nb_annes_ter = 2
df_2023 = nettoyage.calcul_winrate_historique(df_2023,nb_annes_ter)
df_2023["difference winrate home-away "+str(nb_annes_ter)+" ans"] = df_2023["winrate_home "+str(nb_annes_ter)+" ans"]-df_2023["winrate_away "+str(nb_annes_ter)+" ans"]
nb_annes_bis = 3
df_2023 = nettoyage.calcul_winrate_historique(df_2023,nb_annes_bis)
df_2023["difference winrate home-away "+str(nb_annes_bis)+" ans"] = df_2023["winrate_home "+str(nb_annes_bis)+" ans"]-df_2023["winrate_away "+str(nb_annes_bis)+" ans"]
#Ajout évolution de la différence de winrate entre les deux équipes entre long terme et court terme 
df_2023["evolution difference winrate home-away entre "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["difference winrate home-away "+str(nb_annes)+" ans"] - df_2023["difference winrate home-away "+str(nb_annes_bis)+" ans"]

#Ajout winrate confrontation
nb_annes = 1
df_2023 = nettoyage.ajout_winrate_confrontation_directe(df_2023,nb_annes)
nb_annes_ter = 2
df_2023 = nettoyage.ajout_winrate_confrontation_directe(df_2023,nb_annes_ter)
nb_annes_bis = 3
df_2023 = nettoyage.ajout_winrate_confrontation_directe(df_2023,nb_annes_bis)

#Ajout classement moyen des dernières années
nb_annes = 1
df_2023 = nettoyage.ajout_classement_moyen_historique(df_2023,nb_annes)
df_2023["difference classement moyen home-away "+str(nb_annes)+" ans"] = df_2023["pos_moy_home_"+str(nb_annes)+"_ans"]-df_2023["pos_moy_away_"+str(nb_annes)+"_ans"]
nb_annes_ter = 2
df_2023 = nettoyage.ajout_classement_moyen_historique(df_2023,nb_annes_ter)
df_2023["difference classement moyen home-away "+str(nb_annes_ter)+" ans"] = df_2023["pos_moy_home_"+str(nb_annes_ter)+"_ans"]-df_2023["pos_moy_away_"+str(nb_annes_ter)+"_ans"]
nb_annes_bis = 3
df_2023 = nettoyage.ajout_classement_moyen_historique(df_2023,nb_annes_bis)
df_2023["difference classement moyen home-away "+str(nb_annes_bis)+" ans"] = df_2023["pos_moy_home_"+str(nb_annes_bis)+"_ans"]-df_2023["pos_moy_away_"+str(nb_annes_bis)+"_ans"]
#Ajout évolution de classement long terme - court terme
df_2023["evolution classement home "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["pos_moy_home_"+str(nb_annes)+"_ans"] - df_2023["pos_moy_home_"+str(nb_annes_bis)+"_ans"]
df_2023["evolution classement away "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["pos_moy_away_"+str(nb_annes)+"_ans"] - df_2023["pos_moy_away_"+str(nb_annes_bis)+"_ans"]

#Ajout évolution de la différence de classement entre les deux équipes entre long terme et court terme 
df_2023["evolution difference classement entre équipes entre "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["difference classement moyen home-away "+str(nb_annes)+" ans"] - df_2023["difference classement moyen home-away "+str(nb_annes_bis)+" ans"]

#Ajout de la nouveauté d’un club dans la ligue
#df_2023 = nettoyage.ajout_bool_nouveau_club(df_2023)
df_2023 = nettoyage.ajout_bool_nouveau_club_multi_annees(df_2023,1)
df_2023 = nettoyage.ajout_bool_nouveau_club_multi_annees(df_2023,3)

#Ajout de la différence de but moyenne
nb_annes = 1
df_2023 = nettoyage.ajout_diffbut_moyen_historique(df_2023,nb_annes)
df_2023["difference de buts moyenne home-away "+str(nb_annes)+" ans"]= df_2023["diffbut_moy_home_"+str(nb_annes)+"_ans"]-df_2023["diffbut_moy_away_"+str(nb_annes)+"_ans"]
nb_annes_ter = 2
df_2023 = nettoyage.ajout_diffbut_moyen_historique(df_2023,nb_annes_ter)
df_2023["difference de buts moyenne home-away "+str(nb_annes_ter)+" ans"]= df_2023["diffbut_moy_home_"+str(nb_annes_ter)+"_ans"]-df_2023["diffbut_moy_away_"+str(nb_annes_ter)+"_ans"]
nb_annes_bis = 3
df_2023 = nettoyage.ajout_diffbut_moyen_historique(df_2023,nb_annes_bis)
df_2023["difference de buts moyenne home-away "+str(nb_annes_bis)+" ans"]= df_2023["diffbut_moy_home_"+str(nb_annes_bis)+"_ans"]-df_2023["diffbut_moy_away_"+str(nb_annes_bis)+"_ans"]   

#Ajout évolution de la différence de but long terme - court terme
df_2023["evolution difference de buts home "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["diffbut_moy_home_"+str(nb_annes)+"_ans"] - df_2023["diffbut_moy_home_"+str(nb_annes_bis)+"_ans"]
df_2023["evolution difference de buts away "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["diffbut_moy_away_"+str(nb_annes)+"_ans"] - df_2023["diffbut_moy_away_"+str(nb_annes_bis)+"_ans"]

#Ajout évolution de la différence de buts entre les deux équipes entre long terme et court terme 
df_2023["evolution difference de buts entre équipes entre "+str(nb_annes_bis)+" et "+str(nb_annes)+" ans"] = df_2023["difference de buts moyenne home-away "+str(nb_annes)+" ans"] - df_2023["difference de buts moyenne home-away "+str(nb_annes_bis)+" ans"]

#Ajout différence de but moyenne confrontation
nb_annes = 1
df_2023 = nettoyage.ajout_diffbut_moyenne_confrontation_historique(df_2023,nb_annes)
nb_annes_ter = 2
df_2023 = nettoyage.ajout_diffbut_moyenne_confrontation_historique(df_2023,nb_annes_ter)
nb_annes_bis = 3
df_2023 = nettoyage.ajout_diffbut_moyenne_confrontation_historique(df_2023,nb_annes_bis)

df_2023 = df_2023.drop(columns=["away_club_goals","home_club_goals"])
display(df_2023.tail(3))


,home_club_name,away_club_name,game_id,date,season,home_club_id,away_club_id,home_club_position,away_club_position,results,...,difference de buts moyenne home-away 2 ans,diffbut_moy_home_3_ans,diffbut_moy_away_3_ans,difference de buts moyenne home-away 3 ans,evolution difference de buts home 3 et 1 ans,evolution difference de buts away 3 et 1 ans,evolution difference de buts entre équipes entre 3 et 1 ans,diffbut_moy_confrontation_1_ans,diffbut_moy_confrontation_2_ans,diffbut_moy_confrontation_3_ans
4212,Football Club de Nantes,Stade Rennais Football Club,4094845,2024-04-20,2023,995,273,NaN,NaN,NaN,...,-1.092105,-0.166667,0.736842,-0.903509,-0.307018,0.052632,-0.359649,-2.0,-1.0,-0.833333
4255,Le Havre Athletic Club,Football Club de Metz,4094846,2024-04-21,2023,738,347,NaN,NaN,NaN,...,0.894737,0.000000,-0.500000,0.500000,0.000000,0.500000,-0.500000,0.0,0.0,0.000000
4341,Stade brestois 29,Association sportive de Monaco Football Club,4094847,2024-04-21,2023,3911,162,NaN,NaN,NaN,...,-0.723684,-0.298246,0.622807,-0.921053,0.035088,-0.307018,0.342105,-1.0,-0.5,-0.500000


In [ ]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4348 entries, 1491 to 4341
Data columns (total 60 columns):
 #   Column                                                          Non-Null Count  Dtype         
---  ------                                                          --------------  -----         
 0   home_club_name                                                  4348 non-null   object        
 1   away_club_name                                                  4348 non-null   object        
 2   game_id                                                         4348 non-null   int64         
 3   date                                                            4348 non-null   datetime64[ns]
 4   season                                                          4348 non-null   int64         
 5   home_club_id                                                    4348 non-null   int64         
 6   away_club_id                                                    4348 non-null   int64     

In [140]:
df_2023.to_csv("../data/preparee/dataframe_full_2022_2023.csv", index=False)